### Supabase DB 및 Gemini 임베딩을 이용한 RAG 데이터 적재 (v1.2.3)
- 대상 CSV: main_v1_1_8.csv
- 주요 업데이트 내역:
  1. `target_group` 대신 `target_tags`를 임베딩의 대상 키워드로 사용
  2. `target_tags` DB 저장 시 리스트(ARRAY) 형태로 후처리
  3. DB 내 최근 14일 데이터를 조회하여 신규 수집된(DB에 없는) 데이터만 필터링하여 임베딩 및 적재
  4. DB 구조에 맞게 데이터 매핑
  
- 필요한 키 : GOOGLE_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY

In [9]:
import os
import glob
import google.generativeai as genai
import pandas as pd
import json
import time
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv(override=True)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

# 클라이언트 초기화
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

print("환경 설정 및 Supabase 클라이언트 초기화 완료")

환경 설정 및 Supabase 클라이언트 초기화 완료


### 1. DB 기적재 데이터 검증 및 신규 데이터 필터링
- API 수집 주기가 7일이므로, 안전하게 14일 이전까지의 DB 데이터를 불러와 신규 수집된 데이터와 비교합니다.
- DB에 이미 적재된 `(source, source_id)` 세트를 걸러내고, 새롭게 추가된 데이터만 도출합니다.

In [17]:
# 1. DB에서 최근 14일 이내 생성된 (source, source_id) 조합 가져오기
fourteen_days_ago = (datetime.now() - timedelta(days=14)).isoformat()
res = supabase.table('announcements').select('source, source_id').gte('created_dt', fourteen_days_ago).execute()
db_records = res.data

# DB에 존재하는 (source, source_id) 세트 생성
existing_keys = set((row['source'], str(row['source_id'])) for row in db_records)
print(f"최근 14일 내 DB에 기적재된 데이터 건수: {len(existing_keys)}건")

# 2. CSV 파일 로드 (가장 최신 main 데이터)
csv_path = "data/csv/main/main_v1_1_8.csv"
df = pd.read_csv(csv_path)
print(f"CSV 전체 데이터 건수: {len(df)}건")

# 3. DB에 없는 신규 데이터만 필터링
new_df = df[~df.apply(lambda row: (row['source'], str(row['source_id'])) in existing_keys, axis=1)].copy()
print(f"\n🚀 DB에 적재되지 않은 신규 데이터 건수(임베딩 및 적재 대상): {len(new_df)}건")

최근 14일 내 DB에 기적재된 데이터 건수: 0건
CSV 전체 데이터 건수: 547건

🚀 DB에 적재되지 않은 신규 데이터 건수(임베딩 및 적재 대상): 547건


In [18]:
df.tail(5)

,source,source_id,title,summary,s_category,provider,region,target_group,target_tags,target_age,...,required_documents,application_method,detail_url,_scope,_scope_reason,norm_title,norm_provider,norm_period,category,subcategory
542,youth,20260413005400212711,울산형 공공예식장 지원사업,예비부부에게 합리적이고 부담 없는 결혼 기회 제공 및 지역 맞춤형 결혼 지원 정책으...,복지/문화,울산광역시,울산광역시 기업투자국,0013010 | 0049010,청년,만 0세 ~ 만 0세,...,NaN,NaN,https://www.ulsan.go.kr/s/ulsanyouth/bbs/view....,main,primary,울산형공공예식장지원사업,울산광역시,20260301 ~ 20261130~20261130,금융･복지･문화,문화활동 및 생활지원
543,youth,20260413005400212710,미혼 직장남녀 만남 프로그램 운영,저출생 및 인구 문제 대응 방안을 지역사회에 협력·확산하기 위한 울산 직장인 미혼남...,복지/문화,울산광역시,울산광역시 기업투자국,0013010 | 0049010,청년,만 0세 ~ 만 0세,...,NaN,NaN,https://www.ulsan.go.kr/s/ulsanyouth/bbs/view....,main,primary,미혼직장남녀만남프로그램운영,울산광역시,NaN,금융･복지･문화,문화활동 및 생활지원
544,youth,20260413005400212709,청년크루 페스티벌,청년층에 의해 기획·운영되는 문화 향유 행사로 청년들이 즐길수 있는 꿀잼도시 울산 ...,복지/문화,울산광역시,울산광역시 기업투자국,0013010 | 0049010,청년,만 19세 ~ 만 39세,...,NaN,NaN,https://www.ulsan.go.kr/s/ulsanyouth/bbs/view....,main,primary,청년크루페스티벌,울산광역시,20260901 ~ 20260930~,금융･복지･문화,문화활동 및 생활지원
545,youth,20260413005400212708,울산 비보이 페스티벌,젊은이들이 향유하는 힙합문화의 한 장르인 비보잉이 배틀 형식으로\n 진행되는 축제로...,복지/문화,울산광역시,울산광역시 기업투자국,0013010 | 0049010,청년,만 0세 ~ 만 0세,...,NaN,NaN,https://www.ulsan.go.kr/s/ulsanyouth/bbs/view....,main,primary,울산비보이페스티벌,울산광역시,20260701 ~ 20261031~20261031,금융･복지･문화,문화활동 및 생활지원
546,youth,20260413005400212707,울산 청년예술 지원사업,관내 청년예술인의 창작발표 활동 지원을 통한 안정적 예술현장 정착 도모,복지/문화,울산광역시,울산광역시 기업투자국,0013010 | 0049010,청년,만 19세 ~ 만 39세,...,NaN,NaN,https://www.ulsan.go.kr/s/ulsanyouth/bbs/view....,main,primary,울산청년예술지원사업,울산광역시,20260101 ~ 20261231~20261231,금융･복지･문화,예술인지원


### 2. 데이터 전처리 및 임베딩 텍스트 구성
- 기존 `target_group`에 있던 긴 문자열 등의 노이즈를 제거하기 위해, 임베딩 '대상' 컬럼 값을 `target_tags`로 교체합니다.
- 임베딩 텍스트를 만들 때 값이 비어있는 경우 벡터 공간상에서 불필요한 노이즈가 생겨 검색 품질이 미세하게 떨어질 수 있으므로 'target_tags'가 빈값인 경우는 '제한없음'으로 변환하여 임베딩. (db의 target_tags 컬럼은 빈값 그대로 입력)

In [19]:
# 임베딩용 텍스트 구성
def combine_features(row):
    tag = str(row['target_tags']) if pd.notna(row['target_tags']) and str(row['target_tags']).strip() != '' else '제한없음'
    return f"제목: {row['title']}\n카테고리: {row['s_category']}\n지역: {row['region']}\n대상: {tag}\n요약: {row['summary']}"

if len(new_df) > 0:
    new_df['combined_text'] = new_df.apply(combine_features, axis=1)
    display(new_df[['title', 'combined_text']].head(2))

,title,combined_text
0,2026년 고흥군 청년 창업 도전 프로젝트 참여자 모집 연장 공고,제목: 2026년 고흥군 청년 창업 도전 프로젝트 참여자 모집 연장 공고\n카테고리...
1,2026년 2차 배출권거래제 할당대상업체 탄소중립 컨설팅 사업 공고,제목: 2026년 2차 배출권거래제 할당대상업체 탄소중립 컨설팅 사업 공고\n카테고...


### 3. Gemini 임베딩 생성 및 JSON 저장
- 시분초를 제외하고 **날짜(YYYYMMDD)** 기준으로 파일명 생성
- 임베딩이 완료되면 메모리가 아닌 **물리적 JSON 파일로 저장**하여 파이프라인 분리

In [20]:
# 임베딩 생성 함수
def get_embedding(text):
    model_name = "models/gemini-embedding-001"
    result = genai.embed_content(
        model=model_name,
        content=text,
        task_type="retrieval_document",
        output_dimensionality=768
    )
    return result['embedding']

def split_text(text, max_length=1500):
    if len(text) <= max_length:
        return [text]
    return [text[i:i + max_length] for i in range(0, len(text), max_length)]

# 파일명: 시분초 제외, 날짜(YYYYMMDD)만 포함
script_version = "v1_2_3"
today_str = datetime.now().strftime("%Y%m%d") 
embedding_dir = "data/embedding"
embedding_file = os.path.join(embedding_dir, f"embedded_announcements_{script_version}_{today_str}.json")
os.makedirs(embedding_dir, exist_ok=True)

insert_data = []

if len(new_df) > 0:
    print(f"새로운 임베딩을 생성합니다. (API 호출 발생, 대상: {len(new_df)}건)")
    
    for idx, row in new_df.iterrows():
        full_text = row['combined_text']
        chunks = split_text(full_text, max_length=1500)
        
        for chunk in chunks:
            try:
                embedding = get_embedding(chunk)
                data = row.to_dict()
                
                for key, val in data.items():
                    if val == '확인필요' or pd.isna(val):
                        data[key] = None
                
                data["content"] = chunk
                data["embedding"] = embedding
                
                if 'combined_text' in data:
                    del data['combined_text']
                if 'additional_conditions' in data:
                    del data['additional_conditions']
                
                insert_data.append(data)
                time.sleep(0.5)
            except Exception as e:
                print(f"임베딩 생성 오류 (Index {idx}): {e}")
                
    # JSON 파일로만 저장 완료
    with open(embedding_file, 'w', encoding='utf-8') as f:
        json.dump(insert_data, f, ensure_ascii=False, indent=2)
    print(f"✅ 신규 임베딩 데이터가 JSON 파일로 저장되었습니다: {embedding_file}")
else:
    print("신규 임베딩 대상 데이터가 없습니다.")

새로운 임베딩을 생성합니다. (API 호출 발생, 대상: 547건)
✅ 신규 임베딩 데이터가 JSON 파일로 저장되었습니다: data/embedding\embedded_announcements_v1_2_3_20260430.json


### 4. JSON 파일 로드 및 DB 적재 (독립 실행 가능)
- **가장 최근에 저장된 JSON 파일을 자동으로 찾아 로드**합니다.
- 로드된 데이터를 DB 스키마에 맞게 매핑/후처리하고 Supabase에 Insert 합니다.

In [ ]:
# -----------------------------------------------------
# Step A: 저장된 최신 JSON 임베딩 파일 찾아서 읽기
# -----------------------------------------------------
embedding_dir = "data/embedding"
script_version = "v1_2_3"
search_pattern = os.path.join(embedding_dir, f"embedded_announcements_{script_version}_*.json")

file_list = glob.glob(search_pattern)
loaded_data = []

if not file_list:
    print("⚠️ 불러올 임베딩 파일이 없습니다.")
else:
    # 파일 수정 시간 기준으로 가장 최근 파일 선택
    latest_file = max(file_list, key=os.path.getmtime)
    print(f"가장 최근 임베딩 파일을 불러옵니다: {latest_file}")
    
    with open(latest_file, 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    print(f"✅ {len(loaded_data)}건의 데이터를 로드했습니다.")

# -----------------------------------------------------
# Step B: DB 컬럼 매핑 및 후처리 로직
# -----------------------------------------------------
if len(loaded_data) > 0:
    # 1. DB 스키마 조회 (현재 DB에 존재하는 컬럼 목록을 실시간으로 가져와서 매핑되는 값만 리스트업)
    headers = {
        "apikey": SUPABASE_SERVICE_KEY,
        "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}"
    }
    res = requests.get(f"{SUPABASE_URL}/rest/v1/", headers=headers)
    spec = res.json()
    table_columns = set(spec['definitions']['announcements']['properties'].keys())
    
    # 2. 매핑 및 형변환 로직
    rename_map = {
        'apply_start': 'apply_start_dt',
        'apply_end': 'apply_end_dt'
    }
    
    def parse_date(date_val):
        if not date_val: return None
        date_str = str(date_val).strip()
        if len(date_str) == 8 and date_str.isdigit():
            return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
        elif len(date_str) >= 10 and "-" in date_str:
            return date_str
        return None

    filtered_data = []
    
    for data in loaded_data:
        filtered_row = {}
        for k, v in data.items():
            mapped_key = rename_map.get(k, k)
            if mapped_key in table_columns:
                if mapped_key in ['target_age_min', 'target_age_max']:
                    if v is not None:
                        try:
                            filtered_row[mapped_key] = int(float(v))
                        except (ValueError, TypeError):
                            filtered_row[mapped_key] = None
                    else:
                        filtered_row[mapped_key] = None
                elif mapped_key in ['apply_start_dt', 'apply_end_dt']:
                    filtered_row[mapped_key] = parse_date(v)
                elif mapped_key == 'target_tags':
                    if v is not None and str(v).strip() != '':
                        filtered_row[mapped_key] = [tag.strip() for tag in str(v).split(',')]
                    else:
                        filtered_row[mapped_key] = []
                else:
                    filtered_row[mapped_key] = v
        filtered_data.append(filtered_row)
    
    print("✅ 데이터 후처리 및 컬럼 매핑 완료!")
    
    # -----------------------------------------------------
    # Step C: Supabase DB 적재 (Insert) 
    # -> 앞에서 중복필터링을 수행하므로 무거운upsert문대신 isnert문 사용
    # -----------------------------------------------------
    print(f"\n총 {len(filtered_data)}개의 데이터 적재를 시작합니다...")
    batch_size = 1
    
    for i in range(0, len(filtered_data), batch_size):
        batch = filtered_data[i:i + batch_size]
        success = False
        
        for retry in range(5):
            try:
                supabase.table("announcements").insert(batch).execute()
                if (i + 1) % 10 == 0 or (i + 1) == len(filtered_data):
                    print(f"[{i+1}/{len(filtered_data)}] 적재 중...")
                success = True
                break 
            except Exception as e:
                print(f"⚠️ [{i+1}번 데이터] 오류: {e}")
                supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
                time.sleep((retry + 1) * 2)
        
        if not success:
            print(f"❌ {i+1}번 데이터 적재 실패.")
    
    print("\n🎉 DB 적재 완료되었습니다!")

가장 최근 임베딩 파일을 불러옵니다: data/embedding\embedded_announcements_v1_2_3_20260430.json
✅ 547건의 데이터를 로드했습니다.
✅ 데이터 후처리 및 컬럼 매핑 완료!

총 547개의 데이터 적재를 시작합니다...
[10/547] 적재 중...
[20/547] 적재 중...
[30/547] 적재 중...
[40/547] 적재 중...
[50/547] 적재 중...
[60/547] 적재 중...
[70/547] 적재 중...
[80/547] 적재 중...
[90/547] 적재 중...
[100/547] 적재 중...
⚠️ [102번 데이터] 오류: [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2590)
[110/547] 적재 중...
[120/547] 적재 중...
⚠️ [122번 데이터] 오류: [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2590)
[130/547] 적재 중...
[140/547] 적재 중...
[150/547] 적재 중...
[160/547] 적재 중...
[170/547] 적재 중...
[180/547] 적재 중...
[190/547] 적재 중...
[200/547] 적재 중...
[210/547] 적재 중...
[220/547] 적재 중...
[230/547] 적재 중...
[240/547] 적재 중...
[250/547] 적재 중...
[260/547] 적재 중...
[270/547] 적재 중...
[280/547] 적재 중...
[290/547] 적재 중...
[300/547] 적재 중...
⚠️ [303번 데이터] 오류: [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2590)
[310/547] 적재 중...
[320/547] 적재 중...